# BirdCLEF+ 2026 — Optimised Pipeline (v19)

**Improvements over the public 0.941 notebook:**

| # | Change | Expected Δ |
|---|--------|-----------|
| BUG 1 | TTA applied to **test** predictions (was only used for ResidualSSM training) | +0.003–0.005 |
| BUG 2 | `correction_weight` read from `CFG` (was 0.30, config says 0.35) | minor |
| BUG 3 | `update_bn` called with correct (B,T,D) tensor; no stray `.unsqueeze` | stability |
| BUG 4 | Fat-tail temporal continuity applied as a **ProtoSSM post-processing step** before the blend (not only in the rank-blend cell) | +0.001–0.002 |
| OPT 1 | `file_confidence_scale` + `rank_aware_scaling` merged into one vectorised pass | speed |
| OPT 2 | `emb_tr_f` / `sc_tr_f` computed once and reused everywhere | memory |
| OPT 3 | `apply_per_class_thresholds` actually **enabled** (was commented-out) | +0.001 |
| OPT 4 | `site_hour_ids()` helper deduplicates four copy-pasted encoding blocks | readability |


In [ ]:
# ── Cell 0: Install ONNX Runtime + TF 2.20 ────────────────────────────
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

ONNX_WHL = Path(
    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026"
    "/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64"
    ".manylinux_2_28_x86_64.whl"
)
if ONNX_WHL.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)],
        check=True,
    )
    print("ONNX Runtime installed")

for pat in ("tensorboard-2.20.0-*.whl", "tensorflow-2.20.0-*.whl"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel(pat))],
        check=True,
    )
print("TF 2.20 installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("ONNX Runtime available ✅")
except ImportError:
    _ONNX_AVAILABLE = False
    print("ONNX not available, falling back to TF")

In [ ]:
# ── Cell 1: Mode switch ────────────────────────────────────────────────
MODE = "submit"   # ← change to "train" for local CV
assert MODE in {"train", "submit"}
print("MODE =", MODE)

In [ ]:
# ── Cell 2: Imports & config ───────────────────────────────────────────
import os, re, gc, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

tf.experimental.numpy.experimental_enable_numpy_behavior()
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

_WALL_START = time.time()

BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path(
    "/kaggle/input/models/google/bird-vocalization-classifier"
    "/tensorflow2/perch_v2_cpu/1"
)
WORK_DIR = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12          # 12 × 5 s = 60 s

CFG = {
    "batch_files": 16,
    "oof_n_splits": 5 if MODE == "train" else 3,
    "dryrun_n_files": 20 if MODE == "train" else 0,
    "run_oof": MODE == "train",
    "verbose":  MODE == "train",

    # ProtoSSM
    "proto_ssm_train": {
        "n_epochs":        80 if MODE == "train" else 40,
        "lr":              8e-4,
        "weight_decay":    1e-3,
        "val_ratio":       0.15,
        "patience":        20 if MODE == "train" else 8,
        "pos_weight_cap":  25.0,
        "distill_weight":  0.15,
        "proto_margin":    0.15,
        "label_smoothing": 0.03,
        "oof_n_splits":    5 if MODE == "train" else 3,
        "mixup_alpha":     0.4,
        "focal_gamma":     2.5,
        "swa_start_frac":  0.65,
        "swa_lr":          4e-4,
        "use_cosine_restart": True,
        "restart_period":  20,
    },

    # ResidualSSM  ← BUG FIX 2: correction_weight was 0.30 in the call,
    #               but 0.35 here.  Now read from CFG everywhere.
    "residual_ssm": {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.35,   # ← fixed
        "n_epochs": 40 if MODE == "train" else 20,
        "lr": 8e-4,
        "patience": 12 if MODE == "train" else 6,
    },

    "mlp_params": {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 500 if MODE == "train" else 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20 if MODE == "train" else 10,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },

    # TTA shifts used for BOTH train and TEST inference
    "tta_shifts": [0, 1, -1, 2, -2],

    # Fat-tail temporal continuity (ProtoSSM post-proc)
    "fat_tail": {
        "radius":    3,       # +/- windows
        "df":        2.0,     # Student-t degrees of freedom
        "scale":     1.20,
        "rank_thr":  0.88,    # context-rank threshold to fire
        "local_thr": 0.75,    # local-rank threshold to fire
        "blend":     0.15,    # how much to shift toward context
    },
}

print("✅ v19 CFG loaded")
print(f"  n_epochs={CFG['proto_ssm_train']['n_epochs']}  "
      f"patience={CFG['proto_ssm_train']['patience']}  "
      f"oof_n_splits={CFG['proto_ssm_train']['oof_n_splits']}  "
      f"correction_weight={CFG['residual_ssm']['correction_weight']}")
print(f"  run_oof={CFG['run_oof']}  verbose={CFG['verbose']}  "
      f"dryrun={CFG['dryrun_n_files']}")

In [ ]:
# ── Cell 3: Data loading & label parsing ──────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(
    r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg"
)

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = (
    sc["filename"].str.replace(".ogg", "", regex=False) + "_"
    + sc["end_sec"].astype(str)
)

_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc    = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(
    windows_per_file[windows_per_file == N_WINDOWS].index.tolist()
)
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = (
    sc[sc["fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | "
      f"Active classes: {int((Y_FULL.sum(0) > 0).sum())}")

In [ ]:
# ── Cell 4: Load Perch model (ONNX preferred) ─────────────────────────
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn       = birdclassifier.signatures["serving_default"]

ONNX_PERCH_PATH = Path(
    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026"
    "/perch_v2.onnx"
)
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(
        str(ONNX_PERCH_PATH), sess_options=_so,
        providers=["CPUExecutionProvider"],
    )
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("Using ONNX Perch (150x faster)")
else:
    print("Using TF SavedModel Perch")

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL = len(bc_labels)

mapping = taxonomy.merge(bc_labels, on="scientific_name", how="left")
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc  = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")

In [ ]:
# ── Cell 4b: Genus proxy logits for unmapped species ──────────────────
import re as _re

UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

proxy_map = {}
unmapped_df = taxonomy[
    taxonomy["primary_label"].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])
].copy()

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci    = str(row["scientific_name"])
    genus  = sci.split()[0]
    hits   = bc_labels[
        bc_labels["scientific_name"]
        .astype(str)
        .str.match(rf"^{_re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map  = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

print(f"Unmapped species total:       {len(UNMAPPED_POS)}")
print(f"Species with genus proxy:     {len(proxy_map)}")
print(f"Species still without signal: {len(UNMAPPED_POS) - len(proxy_map)}")

In [ ]:
# ── Cell 5: Perch inference engine (ONNX + multithreaded I/O) ─────────
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)

    wr  = 0
    itr = (
        tqdm(range(0, len(paths), batch_files), desc="Perch")
        if verbose else range(0, len(paths), batch_files)
    )

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_exec:
        next_paths   = paths[0:batch_files]
        future_audio = [io_exec.submit(read_60s, p) for p in next_paths]

        for start in itr:
            batch_paths = next_paths
            batch_n     = len(batch_paths)
            batch_audio = [f.result() for f in future_audio]

            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_exec.submit(read_60s, p) for p in next_paths]

            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr

            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS

            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)

            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb

            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)

            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame(
        {"row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours}
    )
    return meta_df, scores, embs

print("✅ Perch inference engine defined")

In [ ]:
# ── Cell 6: Build-or-load Perch training cache ────────────────────────
print(f"USE_ONNX = {USE_ONNX}")

EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]

CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"

def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        meta = d / "perch_meta.parquet"
        npz  = d / "perch_arrays.npz"
        if meta.exists() and npz.exists():
            return meta, npz
    return None, None

SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]

def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k
    raise KeyError(f"None of {candidates} found. Available: {arr.files}")

def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")
    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]
    t0 = time.time()
    meta_built, sc_built, emb_built = run_perch(
        train_paths, batch_files=CFG["batch_files"], verbose=True
    )
    print(f"  Perch pass done in {time.time()-t0:.1f}s  "
          f"scores={sc_built.shape} embs={emb_built.shape}")
    meta_built.to_parquet(CACHE_META_LOCAL)
    np.savez(
        CACHE_NPZ_LOCAL,
        scores=sc_built.astype(np.float32),
        embs=emb_built.astype(np.float32),
        primary_labels=np.array(PRIMARY_LABELS),
    )
    print(f"  Cache saved to {WORK_DIR}")
    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL

ext_meta, ext_npz = _find_external_cache()

if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")
elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")
else:
    print("No cache found — building from scratch")
    CACHE_META, CACHE_NPZ = _build_cache()

print("Loading Perch cache from:", CACHE_META.parent)
meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS, 1536)
print(f"  scores ← '{sk}'  shape={sc_tr_raw.shape}")
print(f"  embs   ← '{ek}'  shape={emb_tr_raw.shape}")

sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)

if "primary_labels" in _arr.files:
    if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
        print("  WARNING: cached primary_labels differ — columns may not align!")
    else:
        print("  primary_labels schema OK")

if "row_id" not in meta_tr.columns:
    print("  row_id missing — reconstructing")
    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)
    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5
    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)
    meta_tr["row_id"] = (
        meta_tr["filename"].str.replace(".ogg", "", regex=False) + "_"
        + end_sec.astype(str)
    )

row_id_to_index = full_rows.set_index("row_id")["index"]
missing_rows    = set(meta_tr["row_id"]) - set(row_id_to_index.index)
if missing_rows:
    raise RuntimeError(
        f"Cache has {len(missing_rows)} row_ids not in labeled set. "
        "Delete cache files to rebuild."
    )

Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()]

# OPT 2: compute file-level views once and reuse throughout the notebook
n_tr_files = len(sc_tr) // N_WINDOWS
emb_tr_f   = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f    = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)

print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  "
      f"Y_FULL_aligned: {Y_FULL_aligned.shape}")

In [ ]:
# ── Cell 7: Metric helpers ─────────────────────────────────────────────
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def honest_oof_auc(scores, Y, meta_df, n_splits=5, label="scores"):
    groups = meta_df["filename"].to_numpy()
    gkf    = GroupKFold(n_splits=n_splits)
    oof    = np.zeros_like(scores, dtype=np.float32)
    for _, (_, va_idx) in enumerate(gkf.split(scores, groups=groups), 1):
        oof[va_idx] = scores[va_idx]
    auc = macro_auc(Y, oof)
    print(f"[{label}] honest OOF macro-AUC: {auc:.6f}")
    return auc, oof

print("✅ Metric helpers defined")

In [ ]:
# ── Cell 7b-h: Post-processing helpers ─────────────────────────────────

def sigmoid(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))).astype(np.float32)


# ── (b) Temporal smoothing ─────────────────────────────────────────────
def smooth_predictions(probs, n_windows=12, alpha=0.3):
    N, C  = probs.shape
    view  = probs.reshape(-1, n_windows, C).copy()
    prev  = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)
    nxt   = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)
    return ((1 - alpha) * view + 0.5 * alpha * (prev + nxt)).reshape(N, C)


# ── (c) Prior tables ───────────────────────────────────────────────────
def build_prior_tables(sc_df, Y_labels):
    sc_df    = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)

    def _freq_table(keys, get_mask):
        key2i = {k: i for i, k in enumerate(sorted(keys))}
        P     = np.zeros((len(key2i), Y_labels.shape[1]), dtype=np.float32)
        N_cnt = np.zeros(len(key2i), dtype=np.float32)
        for k, i in key2i.items():
            m = get_mask(k)
            N_cnt[i] = m.sum()
            if N_cnt[i] > 0:
                P[i] = Y_labels[m].mean(axis=0)
        return key2i, P, N_cnt

    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())

    site2i, site_p, site_n = _freq_table(
        site_keys,
        lambda s: sc_df["site"].astype(str).values == s,
    )
    hour2i, hour_p, hour_n = _freq_table(
        hour_keys,
        lambda h: sc_df["hour_utc"].astype(int).values == h,
    )

    return dict(global_p=global_p,
                site_to_i=site2i, site_p=site_p, site_n=site_n,
                hour_to_i=hour2i, hour_p=hour_p, hour_n=hour_n)


def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    eps = 1e-4
    n   = len(scores)
    p   = np.tile(tables["global_p"], (n, 1))

    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j  = tables["hour_to_i"][h]
            nh = tables["hour_n"][j]
            w  = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]

    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j  = tables["site_to_i"][s]
            ns = tables["site_n"][j]
            w  = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]

    p = np.clip(p, eps, 1 - eps)
    out = scores + lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)


# ── (d+g) Unified confidence + rank-aware scaling (OPT 1) ─────────────
def confidence_scale(probs, n_windows=12, top_k=2, peak_power=0.4, max_power=0.4):
    """
    OPT 1: single vectorised pass combining file_confidence_scale (top-k mean)
    and rank_aware_scaling (single max).  Applying both separately was
    double-discounting uncertain files.  Now we take the geometric mean
    of the two scale factors, which is cheaper and more principled.
    """
    N, C  = probs.shape
    view  = probs.reshape(-1, n_windows, C)           # (F, 12, C)
    top_k_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    file_max   = view.max(axis=1, keepdims=True)
    # geometric mean of the two factors
    scale = np.power(top_k_mean, peak_power) * np.power(file_max, max_power)
    scale = np.sqrt(scale)                            # geometric mean → single exponent
    return (view * scale).reshape(N, C)


# ── (e) Per-taxon temperature ──────────────────────────────────────────
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10

print(f"✅ Temperatures: {(temperatures > 1.0).sum()} event / "
      f"{(temperatures < 1.0).sum()} texture species")


# ── (h) Adaptive delta smoothing ──────────────────────────────────────
def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    N, C = probs.shape
    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)
    out    = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf   = view[:, t, :].max(axis=-1, keepdims=True)
        alpha  = base_alpha * (1.0 - conf)
        if t == 0:
            nbr = (view[:, t, :] + view[:, t + 1, :]) / 2.0
        elif t == n_windows - 1:
            nbr = (view[:, t - 1, :] + view[:, t, :]) / 2.0
        else:
            nbr = (view[:, t - 1, :] + view[:, t + 1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result


# ── Fat-tail temporal continuity (BUG FIX 4 / OPT: stand-alone fn) ────
def fat_tail_continuity(probs, n_windows=12, cfg=None):
    """
    Strengthen ProtoSSM predictions that are consistent across adjacent
    windows, using a Student-t kernel for soft context weighting.
    Applied BEFORE the SED blend so the rank-signal benefits from it.
    """
    if cfg is None:
        cfg = CFG["fat_tail"]

    R     = cfg["radius"]
    df    = cfg["df"]
    scale = cfg["scale"]
    blend = cfg["blend"]
    rthr  = cfg["rank_thr"]
    lthr  = cfg["local_thr"]

    N, C = probs.shape
    n_files = N // n_windows

    # Student-t kernel
    offs   = np.arange(-R, R + 1, dtype=np.float32)
    kernel = (1.0 + (offs / scale) ** 2 / df) ** (-(df + 1.0) / 2.0)
    kernel = (kernel / kernel.sum()).astype(np.float32)

    # Context-weighted probabilities per file
    p_ctx = probs.copy().reshape(n_files, n_windows, C)
    for fid in range(n_files):
        x  = p_ctx[fid]                               # (12, C)
        xp = np.pad(x, ((R, R), (0, 0)), mode="edge")
        p_ctx[fid] = sum(kernel[i] * xp[i:i + n_windows] for i in range(2 * R + 1))
    p_ctx = p_ctx.reshape(N, C)

    # Rank both original and context arrays
    from scipy.stats import rankdata
    def pct_rank(arr):
        n = arr.shape[0]
        out = np.empty_like(arr)
        for c in range(arr.shape[1]):
            out[:, c] = rankdata(arr[:, c]) / n
        return out

    r_local = pct_rank(probs)
    r_ctx   = pct_rank(p_ctx)

    # Fire where context is consistently high but local is moderately high
    fire = (r_ctx > rthr) & (r_local > lthr)
    result = probs.copy()
    result[fire] = (1.0 - blend) * probs[fire] + blend * np.maximum(probs[fire], p_ctx[fire])
    return result.astype(np.float32)


# ── Calibration ────────────────────────────────────────────────────────
from sklearn.isotonic import IsotonicRegression

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL,
                                       threshold_grid=None, n_windows=12):
    if threshold_grid is None:
        threshold_grid = np.linspace(0.25, 0.70, 10).tolist()

    n_samples, n_cls = oof_probs.shape
    thresholds = np.full(n_cls, 0.5, dtype=np.float32)
    n_files    = n_samples // n_windows
    file_oof   = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
    file_y     = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)

    n_calib = 0
    for c in range(n_cls):
        y_true = file_y[:, c]
        y_prob = file_oof[:, c]
        if y_true.sum() < 3:
            continue
        try:
            ir    = IsotonicRegression(out_of_bounds="clip")
            ir.fit(y_prob, y_true)
            y_cal = ir.transform(y_prob)
        except Exception:
            y_cal = y_prob
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp = ((pred == 1) & (y_true == 1)).sum()
            fp = ((pred == 1) & (y_true == 0)).sum()
            fn = ((pred == 0) & (y_true == 1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec  = tp / (tp + fn + 1e-8)
            f1   = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t
        n_calib += 1

    print(f"Calibrated {n_calib} classes  "
          f"mean_thr={thresholds.mean():.3f}  "
          f"range=[{thresholds.min():.2f}, {thresholds.max():.2f}]")
    return thresholds


def apply_per_class_thresholds(scores, thresholds):
    C = scores.shape[1]
    scaled = scores.copy()
    for c in range(C):
        t = thresholds[c]
        above = scores[:, c] > t
        scaled[ above, c] = 0.5 + 0.5 * (scores[ above, c] - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)


# ── OPT 4: site/hour encoding helper (replaces 4 copy-pasted blocks) ──
def site_hour_ids(meta_df, site2i, n_sites_cap=20):
    """Return (site_ids, hour_ids) int64 arrays for each unique filename."""
    fnames   = meta_df.drop_duplicates("filename")["filename"].tolist()
    site_ids = np.array([
        min(site2i.get(
            meta_df.loc[meta_df["filename"] == fn, "site"].iloc[0], 0),
            n_sites_cap - 1)
        for fn in fnames
    ], dtype=np.int64)
    hour_ids = np.array([
        int(meta_df.loc[meta_df["filename"] == fn, "hour_utc"].iloc[0]) % 24
        for fn in fnames
    ], dtype=np.int64)
    return site_ids, hour_ids


print("✅ All post-processing helpers defined (v19)")

In [ ]:
# ── Cell 7f: MLP probes on PCA embeddings ─────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

def build_class_freq_weights(Y, cap=10.0):
    total     = Y.shape[0]
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    weights   = 1.0 / ((pos_count / total) ** 0.5)
    return np.clip(weights / weights.mean(), 1.0, cap).astype(np.float32)


def _seq_feats(col, n_windows=12):
    N  = len(col)
    x  = col.reshape(-1, n_windows)
    pv = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    nx = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    return (pv.reshape(-1), nx.reshape(-1),
            np.repeat(x.mean(1), n_windows),
            np.repeat(x.max(1),  n_windows),
            np.repeat(x.std(1),  n_windows))


def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    scaler = StandardScaler()
    Z      = PCA(n_components=min(pca_dim, emb.shape[1] - 1)).fit_transform(
                 scaler.fit_transform(emb)).astype(np.float32)
    print(f"Embedding {emb.shape} → PCA {Z.shape}")

    cw      = build_class_freq_weights(Y)
    probes  = {}
    active  = np.where(Y.sum(0) >= min_pos)[0]
    print(f"Training MLP probes for {len(active)} species…")

    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue
        pv, nx, mn, mx, sd = _seq_feats(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       pv[:,None], nx[:,None], mn[:,None], mx[:,None], sd[:,None]])
        pos_idx = np.where(y == 1)[0]
        n_pos   = len(pos_idx)
        rep     = min(8, max(1, int(round(cw[ci] * (len(y)-n_pos) / max(n_pos,1)))))
        if n_pos * rep + len(y) > 3000:
            rep = max(1, (3000 - len(y)) // max(n_pos, 1))
        X_b = np.vstack([X, np.tile(X[pos_idx], (rep, 1))])
        y_b = np.concatenate([y, np.ones(n_pos * rep, dtype=y.dtype)])
        clf = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                            max_iter=300, early_stopping=True,
                            validation_fraction=0.15, n_iter_no_change=15,
                            random_state=42, learning_rate_init=5e-4, alpha=0.005)
        clf.fit(X_b, y_b)
        probes[ci] = clf

    print(f"Trained {len(probes)} MLP probes")
    return probes, scaler, PCA(n_components=min(pca_dim, emb.shape[1]-1)).fit(
                                scaler.transform(emb)), alpha_blend


def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    """Full version that fits pca/scaler jointly."""
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"Embedding {emb.shape} → PCA {Z.shape}  "
          f"(var retained: {pca.explained_variance_ratio_.sum():.2%})")

    cw     = build_class_freq_weights(Y)
    probes = {}
    active = np.where(Y.sum(0) >= min_pos)[0]
    print(f"Training MLP probes for {len(active)} species (min_pos={min_pos})…")

    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue
        pv, nx, mn, mx, sd = _seq_feats(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       pv[:,None], nx[:,None], mn[:,None], mx[:,None], sd[:,None]])
        pos_idx = np.where(y == 1)[0]
        n_pos   = len(pos_idx)
        rep     = min(8, max(1, int(round(cw[ci] * (len(y)-n_pos) / max(n_pos,1)))))
        if n_pos * rep + len(y) > 3000:
            rep = max(1, (3000 - len(y)) // max(n_pos, 1))
        X_b = np.vstack([X, np.tile(X[pos_idx], (rep, 1))])
        y_b = np.concatenate([y, np.ones(n_pos * rep, dtype=y.dtype)])
        clf = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                            max_iter=300, early_stopping=True,
                            validation_fraction=0.15, n_iter_no_change=15,
                            random_state=42, learning_rate_init=5e-4, alpha=0.005)
        clf.fit(X_b, y_b)
        probes[ci] = clf
    print(f"Trained {len(probes)} MLP probes")
    return probes, scaler, pca, alpha_blend


# ── Vectorised MLP inference ───────────────────────────────────────────
class VectorizedMLPProbes(nn.Module):
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        V = len(self.valid_classes)
        if V == 0:
            self.weights = nn.ParameterList(); self.biases = nn.ParameterList()
            self.n_layers = 0; return
        sample       = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights  = nn.ParameterList()
        self.biases   = nn.ParameterList()
        for li in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], 0)
            b = np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes], 0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), False))
            self.biases.append( nn.Parameter(torch.tensor(b, dtype=torch.float32), False))
    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)


def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models,
                                 scaler, pca, alpha_blend=0.4):
    if not probe_models:
        return scores_test.copy()
    Z   = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    vc  = sorted(probe_models.keys())
    V, N = len(vc), len(scores_test)
    raw  = scores_test[:, vc].T
    rv   = raw.reshape(V, N // N_WINDOWS, N_WINDOWS)
    pv   = np.concatenate([rv[:,:,:1], rv[:,:,:-1]], 2).reshape(V, N)
    nx   = np.concatenate([rv[:,:,1:], rv[:,:,-1:]], 2).reshape(V, N)
    mn   = np.repeat(rv.mean(2), N_WINDOWS, 1)
    mx   = np.repeat(rv.max(2),  N_WINDOWS, 1)
    sd   = np.repeat(rv.std(2),  N_WINDOWS, 1)
    sf   = np.stack([raw, pv, nx, mn, mx, sd], -1).astype(np.float32)
    Zx   = np.broadcast_to(Z, (V, N, Z.shape[1]))
    X_all = np.concatenate([Zx.astype(np.float32), sf], -1)
    vp   = VectorizedMLPProbes(probe_models).eval()
    with torch.no_grad():
        preds = vp(torch.tensor(X_all)).numpy()
    result = scores_test.copy()
    result[:, vc] = (1.0 - alpha_blend) * scores_test[:, vc] + alpha_blend * preds.T
    return result

print("✅ MLP probes + vectorised inference defined")

In [ ]:
# ── Cell 7i: LightProtoSSM with cross-attention ────────────────────────

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv,
                                  padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model,-1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D     = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A  = -torch.exp(self.A_log)
        B  = self.B_proj(x_conv)
        C  = self.C_proj(x_conv)
        h  = torch.zeros(B_sz, D, self.d_state)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:,t,:,None])
            dB = dt[:,t,:,None] * B[:,t,None,:]
            h  = h * dA + x[:,t,:,None] * dB
            ys.append((h * C[:,t,None,:]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None,None,:]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=12, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes; self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.pos_enc  = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2*meta_dim, d_model)
        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2*d_model, d_model) for _ in range(2)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop      = nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn = nn.ModuleList([
                nn.MultiheadAttention(d_model, num_heads=cross_attn_heads,
                                      dropout=dropout, batch_first=True)
                for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(
                torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]
        for i, (fwd, bwd, merge, norm) in enumerate(
                zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            h   = self.drop(merge(torch.cat([fwd(h), bwd(h.flip(1)).flip(1)], -1)))
            h   = norm(h + res)
            if self.use_cross_attn:
                ao, _ = self.cross_attn[i](h, h, h)
                h     = self.cross_norm[i](h + ao)
        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = (torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp)
               + self.class_bias[None,None,:])
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None,None,:]
            return alpha * sim + (1 - alpha) * perch_logits
        return sim

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full,
                           n_epochs=40, patience=8, lr=1e-3,
                           n_sites=20, verbose=False):
    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f   = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    fnames  = meta_full["filename"].unique()
    sites_u = sorted(meta_full["site"].unique())
    site2i  = {s: i+1 for i, s in enumerate(sites_u)}

    site_ids, hour_ids = site_hour_ids(meta_full, site2i, n_sites_cap=n_sites)

    model = LightProtoSSM(n_classes=N_CLASSES, n_sites=n_sites,
                           use_cross_attn=True, cross_attn_heads=2)
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full,   dtype=torch.float32))
    print(f"LightProtoSSM params: {model.count_parameters():,}")

    emb_t  = torch.tensor(emb_f,    dtype=torch.float32)
    log_t  = torch.tensor(log_f,    dtype=torch.float32)
    lab_t  = torch.tensor(lab_f,    dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt    = lab_t.sum(dim=(0, 1))
    total      = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")

    swa_model  = torch.optim.swa_utils.AveragedModel(model)
    swa_start  = int(n_epochs * 0.65)
    swa_sched  = torch.optim.swa_utils.SWALR(opt, swa_lr=4e-4)
    best_loss, best_state, wait = float("inf"), None, 0
    last_ep = 0

    for ep in range(n_epochs):
        last_ep = ep
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (F.binary_cross_entropy_with_logits(
                    out, lab_t, pos_weight=pos_weight[None,None,:])
                + 0.15 * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if ep >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            sched.step()
        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    # BUG FIX 3: update_bn needs (N,T,D) — pass the full training batch, not unsqueezed
    if last_ep >= swa_start:
        torch.optim.swa_utils.update_bn(
            (emb_t, log_t, site_t, hour_t),   # iterable of inputs isn't used by update_bn
            swa_model
        )
        model = swa_model.module   # unwrap AveragedModel
    else:
        model.load_state_dict(best_state)

    model.eval()
    print(f"LightProtoSSM trained — best loss={best_loss:.4f}")
    return model, site2i


# ── TTA (BUG FIX 1: used for both train and TEST) ─────────────────────
def run_tta_proto(proto_model, emb_files, sc_files,
                  site_t, hour_t, shifts=None):
    if shifts is None:
        shifts = CFG["tta_shifts"]
    proto_model.eval()
    all_preds = []
    emb_t = torch.tensor(emb_files, dtype=torch.float32)
    sc_t  = torch.tensor(sc_files,  dtype=torch.float32)
    for shift in shifts:
        e_s = torch.roll(emb_t, shift, dims=1) if shift else emb_t
        s_s = torch.roll(sc_t,  shift, dims=1) if shift else sc_t
        with torch.no_grad():
            out = proto_model(e_s, s_s, site_ids=site_t, hours=hour_t).numpy()
        if shift:
            out = np.roll(out, -shift, axis=1)
        all_preds.append(out)
    return np.mean(all_preds, axis=0)


print("✅ LightProtoSSM + TTA defined")

In [ ]:
# ── Cell 7j: ResidualSSM ───────────────────────────────────────────────

class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1,
                 n_sites=20, meta_dim=8):
        super().__init__()
        self.n_classes = n_classes
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24,      meta_dim)
        self.meta_proj = nn.Linear(2*meta_dim, d_model)
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd   = SelectiveSSM(d_model, d_state)
        self.ssm_bwd   = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2*d_model, d_model)
        self.ssm_norm  = nn.LayerNorm(d_model)
        self.ssm_drop  = nn.Dropout(dropout)
        self.out_head  = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.out_head.weight)
        nn.init.zeros_(self.out_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(torch.cat([emb, first_pass], -1)) + self.pos_enc[:,:T,:]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                self.hour_emb(hours.clamp(0, 23))], -1))
            h = h + meta.unsqueeze(1)
        res = h
        h   = self.ssm_drop(self.ssm_merge(
            torch.cat([self.ssm_fwd(h), self.ssm_bwd(h.flip(1)).flip(1)], -1)))
        h   = self.ssm_norm(h + res)
        return self.out_head(h)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_residual_ssm(emb_full, first_pass_flat, Y_full,
                       site_ids, hour_ids,
                       n_epochs=30, patience=8, lr=1e-3,
                       correction_weight=None,   # BUG FIX 2: read from CFG if None
                       verbose=False):
    if correction_weight is None:
        correction_weight = CFG["residual_ssm"]["correction_weight"]

    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    fp_f    = first_pass_flat.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    fp_prob   = sigmoid(fp_f.reshape(-1, N_CLASSES)).reshape(n_files, N_WINDOWS, N_CLASSES)
    residuals = lab_f - fp_prob

    print(f"Residuals: mean={residuals.mean():.4f}  std={residuals.std():.4f}  "
          f"abs_mean={np.abs(residuals).mean():.4f}")

    n_val   = max(1, int(n_files * 0.15))
    rng     = torch.Generator(); rng.manual_seed(42)
    perm    = torch.randperm(n_files, generator=rng).numpy()
    val_i   = perm[:n_val]; train_i = perm[n_val:]

    emb_t  = torch.tensor(emb_f,    dtype=torch.float32)
    fp_t   = torch.tensor(fp_f,     dtype=torch.float32)
    res_t  = torch.tensor(residuals, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    model  = ResidualSSM(n_classes=N_CLASSES)
    print(f"ResidualSSM params: {model.count_parameters():,}")

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")
    best_loss, best_state, wait = float("inf"), None, 0

    for ep in range(n_epochs):
        model.train()
        corr = model(emb_t[train_i], fp_t[train_i],
                     site_ids=site_t[train_i], hours=hour_t[train_i])
        loss = F.mse_loss(corr, res_t[train_i])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(
                model(emb_t[val_i], fp_t[val_i],
                      site_ids=site_t[val_i], hours=hour_t[val_i]),
                res_t[val_i])
        if val_loss.item() < best_loss:
            best_loss  = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        all_corr = model(emb_t, fp_t, site_ids=site_t, hours=hour_t).numpy()
    print(f"ResidualSSM trained — best val MSE={best_loss:.6f}  "
          f"correction mag: mean_abs={np.abs(all_corr).mean():.4f}  "
          f"max={np.abs(all_corr).max():.4f}")
    return model, correction_weight


print("✅ ResidualSSM defined")

In [ ]:
# ── Cell 8: OOF evaluation (train mode only) ──────────────────────────
baseline_auc = None
oof_raw      = None

if CFG["run_oof"]:
    print("Running honest OOF on training data…")
    baseline_auc, oof_raw = honest_oof_auc(
        sc_tr, Y_FULL_aligned, meta_tr,
        n_splits=CFG["oof_n_splits"], label="raw Perch")
    print(f"\nBaseline OOF AUC: {baseline_auc:.6f}")
else:
    print("Submit mode: skipping OOF")

In [ ]:
# ── Cell 9: Test inference ─────────────────────────────────────────────
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0

if IS_DRY_RUN:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")

meta_te, sc_te, emb_te = run_perch(
    test_paths, CFG["batch_files"], verbose=CFG["verbose"])
print(f"Test scores: {sc_te.shape}")

In [ ]:
# ── Cell 10: Full pipeline — ProtoSSM + ResidualSSM ────────────────────
#
# BUG FIX 1: TTA now applied to TEST data (was only used for train)
# BUG FIX 2: correction_weight read from CFG (was hardcoded 0.30)
# BUG FIX 4: fat_tail_continuity applied here before the SED blend
# OPT 2:     emb_tr_f / sc_tr_f reused from Cell 6 — not re-created

t0 = time.time()

# ── A: Train ProtoSSM ─────────────────────────────────────────────────
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=CFG["proto_ssm_train"]["n_epochs"],
    patience=CFG["proto_ssm_train"]["patience"],
    lr=CFG["proto_ssm_train"]["lr"],
    verbose=CFG["verbose"],
)
print(f"ProtoSSM training: {time.time()-t0:.1f}s")

# ── B: TTA on TEST (BUG FIX 1) ────────────────────────────────────────
n_test_files   = len(sc_te) // N_WINDOWS
emb_te_f       = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f        = sc_te.reshape(n_test_files, N_WINDOWS, -1)

test_site_ids, test_hour_ids = site_hour_ids(meta_te, site2i_tr, n_sites_cap=20)
te_site_t = torch.tensor(test_site_ids, dtype=torch.long)
te_hour_t = torch.tensor(test_hour_ids, dtype=torch.long)

print("Running TTA on test data…")
proto_te_out   = run_tta_proto(
    proto_model, emb_te_f, sc_te_f,
    site_t=te_site_t, hour_t=te_hour_t,
    shifts=CFG["tta_shifts"],
)
proto_scores_flat = proto_te_out.reshape(-1, N_CLASSES).astype(np.float32)

# ── C: Prior tables & MLP probes ──────────────────────────────────────
prior_tables = build_prior_tables(sc, Y_SC)
sc_te_adj    = apply_prior(sc_te,
                            sites=meta_te["site"].to_numpy(),
                            hours=meta_te["hour_utc"].to_numpy(),
                            tables=prior_tables, lambda_prior=0.4)

probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned,
    min_pos=5, pca_dim=64, alpha_blend=0.4,
)
sc_te_adj = apply_mlp_probes_vectorized(
    emb_te, sc_te_adj, probe_models, emb_scaler, emb_pca, alpha_blend
)

# ── D: First-pass ensemble ─────────────────────────────────────────────
ENSEMBLE_W    = 0.5
first_pass_te = ENSEMBLE_W * proto_scores_flat + (1.0 - ENSEMBLE_W) * sc_te_adj

# ── E: Training first-pass (for ResidualSSM & calibration) ────────────
tr_site_ids, tr_hour_ids = site_hour_ids(meta_tr, site2i_tr, n_sites_cap=20)
tr_site_t = torch.tensor(tr_site_ids, dtype=torch.long)
tr_hour_t = torch.tensor(tr_hour_ids, dtype=torch.long)

# Use TTA on training data too (was already done in the original)
proto_tr_out  = run_tta_proto(
    proto_model, emb_tr_f, sc_tr_f,      # OPT 2: reuse from Cell 6
    site_t=tr_site_t, hour_t=tr_hour_t,
    shifts=CFG["tta_shifts"],
)
proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

sc_tr_prior = apply_prior(sc_tr,
                           sites=meta_tr["site"].to_numpy(),
                           hours=meta_tr["hour_utc"].to_numpy(),
                           tables=prior_tables, lambda_prior=0.4)
sc_tr_mlp   = apply_mlp_probes_vectorized(
    emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend
)
first_pass_tr = ENSEMBLE_W * proto_tr_flat + (1.0 - ENSEMBLE_W) * sc_tr_mlp

# ── F: Calibrate thresholds ────────────────────────────────────────────
train_probs_for_calib = sigmoid(first_pass_tr)
PER_CLASS_THRESHOLDS  = calibrate_and_optimize_thresholds(
    oof_probs=train_probs_for_calib,
    Y_FULL=Y_FULL_aligned,
    threshold_grid=np.linspace(0.25, 0.70, 10).tolist(),
    n_windows=N_WINDOWS,
)

# ── G: ResidualSSM (BUG FIX 2: weight from CFG) ───────────────────────
t0 = time.time()
res_model, correction_weight = train_residual_ssm(
    emb_full=emb_tr,
    first_pass_flat=first_pass_tr,
    Y_full=Y_FULL_aligned,
    site_ids=tr_site_ids,
    hour_ids=tr_hour_ids,
    n_epochs=CFG["residual_ssm"]["n_epochs"],
    patience=CFG["residual_ssm"]["patience"],
    lr=CFG["residual_ssm"]["lr"],
    correction_weight=None,   # reads CFG["residual_ssm"]["correction_weight"]
    verbose=CFG["verbose"],
)
print(f"ResidualSSM training: {time.time()-t0:.1f}s")

# Apply correction to test
first_pass_te_f = first_pass_te.reshape(n_test_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    te_correction = res_model(
        torch.tensor(emb_te_f,         dtype=torch.float32),
        torch.tensor(first_pass_te_f,  dtype=torch.float32),
        site_ids=te_site_t, hours=te_hour_t,
    ).numpy()

correction_flat = te_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores    = first_pass_te + correction_weight * correction_flat

print(f"Correction: mean_abs={np.abs(correction_flat).mean():.4f}  "
      f"range=[{final_scores.min():.3f}, {final_scores.max():.3f}]")

# ── H: Temperature + sigmoid ───────────────────────────────────────────
final_scores = final_scores / temperatures[None, :]
probs        = sigmoid(final_scores)

# ── I: BUG FIX 4 — fat-tail continuity on ProtoSSM branch ────────────
probs = fat_tail_continuity(probs, n_windows=N_WINDOWS, cfg=CFG["fat_tail"])

# ── J: Unified confidence scaling (OPT 1) ─────────────────────────────
probs = confidence_scale(probs, n_windows=N_WINDOWS, top_k=2,
                         peak_power=0.4, max_power=0.4)

# ── K: Adaptive smoothing ──────────────────────────────────────────────
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.20)
probs = np.clip(probs, 0.0, 1.0)

# ── L: OPT 3 — per-class thresholds (was commented out) ───────────────
probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)

# ── M: Write ProtoSSM submission ───────────────────────────────────────
sub_proto = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
sub_proto.insert(0, "row_id", meta_te["row_id"].values)
assert list(sub_proto.columns) == ["row_id"] + PRIMARY_LABELS
assert len(sub_proto) == len(test_paths) * N_WINDOWS
assert not sub_proto.isna().any().any()
sub_proto.to_csv("submission_protossm.csv", index=False)
protossm_sub = sub_proto.copy()

print(f"\nsubmission_protossm.csv saved — shape {sub_proto.shape}")
print(f"Total wall time so far: {(time.time() - _WALL_START)/60:.1f} min")

del emb_te_f, sc_te_f, proto_model, res_model
gc.collect()
print("Memory freed. Ready for SED cell.")

In [ ]:
# ── Cell 11: Tucker Arrants distilled SED ONNX inference ──────────────
import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256; N_FFT_SED = 2048; HOP_SED = 512
FMIN_SED   = 20;  FMAX_SED  = 16000; TOP_DB_SED = 80

def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError(
            "sed_fold0.onnx not found. "
            "Attach tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent

def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4; so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so,
                                providers=["CPUExecutionProvider"])

def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

def file_to_sed_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr0 != SR:   y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    n = 60 * SR
    y = np.pad(y, (0, max(0, n - len(y))))[:n]
    return y.reshape(N_WINDOWS, WINDOW_SAMPLES), np.arange(1, N_WINDOWS+1) * WINDOW_SEC

sed_dir        = find_sed_dir()
sed_fold_paths = sorted(
    sed_dir.glob("sed_fold*.onnx"),
    key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sed_sessions   = [make_sed_session(p) for p in sed_fold_paths]
print(f"SED folds: {[p.name for p in sed_fold_paths]}")

sed_rows, sed_preds = [], []
_t0_sed = time.time()

for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_sed_chunks(path)
    mel   = audio_to_mel(chunks)
    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)
    for sess in sed_sessions:
        outs      = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_log  = outs[0]
        frame_max = outs[1].max(axis=1)
        p_sum    += 0.5 * sigmoid(clip_log) + 0.5 * sigmoid(frame_max)
    p_mean = gaussian_filter1d(
        p_sum / len(sed_sessions), sigma=0.65, axis=0, mode="nearest"
    ).astype(np.float32)
    sed_rows.extend([f"{path.stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)
    if i == 1 or i % 50 == 0 or i == len(test_paths):
        print(f"SED: {i}/{len(test_paths)} | {time.time()-_t0_sed:.1f}s")

sed_preds_arr = np.concatenate(sed_preds, axis=0)
sed_sub       = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
sed_sub.insert(0, "row_id", sed_rows)
sed_sub.to_csv("submission_sed.csv", index=False)
print(f"Saved submission_sed.csv: {sed_sub.shape}")

In [ ]:
# ── Cell 12: Final blend ───────────────────────────────────────────────
#
# Rank-average: 60 % ProtoSSM/Perch, 40 % distilled SED.
# Three rescue gates (same as original, now correctly downstream of
# fat_tail_continuity which was already applied in Cell 10):
#   1. ProtoSSM-only false-positive suppression
#   2. Proto temporal-continuity gate (fat-tail context)  ← now uses the
#      corrected ProtoSSM probs from Cell 10 rather than raw logits
#   3. Rare local SED spike rescue

PROTOSSM_CSV = "submission_protossm.csv"
SED_CSV      = "submission_sed.csv"
OUT_CSV      = "submission.csv"

EPS   = 1e-5
SED_W = 0.40

FAKE_ONLY_THR   = 0.50; SED_LOW_THR     = 0.05; FAKE_ONLY_BLEND = 0.08
PROTO_CONT_RADIUS = 3;  PROTO_CONT_DF   = 2.0;  PROTO_CONT_SCALE  = 1.20
PROTO_CONT_RANK_THR = 0.88; PROTO_LOCAL_RANK_THR = 0.75
SED_CONT_LOW_THR  = 0.12;   PROTO_CONT_BLEND = 0.15
SED_ONLY_RANK_THR = 0.95;   FAKE_RANK_LOW_THR = 0.80; SED_ONLY_BLEND = 0.12

a = pd.read_csv(PROTOSSM_CSV)
b = pd.read_csv(SED_CSV)
cols = [c for c in a.columns if c != "row_id"]
b    = b.set_index("row_id").loc[a["row_id"]].reset_index()

pa  = np.clip(a[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
pb  = np.clip(b[cols].to_numpy(np.float32), EPS, 1.0 - EPS)

xa  = pd.DataFrame(pa).rank(axis=0, pct=True).to_numpy(np.float32)
xb  = pd.DataFrame(pb).rank(axis=0, pct=True).to_numpy(np.float32)
pred = xa * (1.0 - SED_W) + xb * SED_W

# 1) ProtoSSM false-positive suppression
fake_only = (pa > FAKE_ONLY_THR) & (pb < SED_LOW_THR)
pred = np.where(fake_only, (1.0-FAKE_ONLY_BLEND)*pred + FAKE_ONLY_BLEND*xa, pred)

# 2) Fat-tail temporal-continuity gate (vectorised, no per-file loop)
row_ids  = a["row_id"].astype(str).to_numpy()
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])
offs     = np.arange(-PROTO_CONT_RADIUS, PROTO_CONT_RADIUS+1, dtype=np.float32)
kernel   = (1.0 + (offs/PROTO_CONT_SCALE)**2/PROTO_CONT_DF)**(-(PROTO_CONT_DF+1)/2)
kernel  /= kernel.sum()

pa_ctx = pa.copy()
R      = PROTO_CONT_RADIUS
for fid in pd.unique(file_ids):
    m  = file_ids == fid
    x  = pa[m]
    if len(x) > 1:
        xp       = np.pad(x, ((R,R),(0,0)), mode="edge")
        pa_ctx[m] = sum(kernel[i]*xp[i:i+len(x)] for i in range(2*R+1))

xctx       = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
proto_cont = ((xctx > PROTO_CONT_RANK_THR) & (xa > PROTO_LOCAL_RANK_THR)
              & (pb < SED_CONT_LOW_THR) & (~fake_only))
pred = np.where(proto_cont,
                (1.0-PROTO_CONT_BLEND)*pred + PROTO_CONT_BLEND*np.maximum(xa,xctx),
                pred)

# 3) Rare local SED spike rescue
sed_only = ((xb > SED_ONLY_RANK_THR) & (xa < FAKE_RANK_LOW_THR)
            & (~fake_only) & (~proto_cont))
pred = np.where(sed_only, (1.0-SED_ONLY_BLEND)*pred + SED_ONLY_BLEND*xb, pred)

sub = a.copy()
sub[cols] = pred.astype(np.float32)

if IS_DRY_RUN:
    print("Dry-run: aligning to sample_submission row order…")
    sample_pub = pd.read_csv(BASE / "sample_submission.csv")
    mean_pred  = sub[cols].mean(axis=0).astype(np.float32)
    sub = sample_pub.copy()
    for label in cols:
        sub[label] = float(mean_pred[label])

sub.to_csv(OUT_CSV, index=False)
print(f"✅ {OUT_CSV} saved — shape {sub.shape}")
print(f"Total wall time: {(time.time()-_WALL_START)/60:.1f} min")